# SEND MESSAGES

This code cell sends a test message to the Telegram API to verify connectivity and retrieve updates. It's used for initial setup and debugging of the Telegram bot's communication.


In [ ]:
import requests
from google.colab import userdata

# Retrieve Telegram Bot Token from Colab secrets
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
    if not TELEGRAM_BOT_TOKEN:
        raise ValueError("TELEGRAM_BOT_TOKEN found but is empty. Please ensure your bot token is correctly set.")
    print("Successfully loaded TELEGRAM_BOT_TOKEN for getUpdates test.")
except userdata.SecretNotFoundError:
    raise ValueError("TELEGRAM_BOT_TOKEN not found. Please add your Telegram bot token to Colab secrets.")

base_url=f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getUpdates"
parameters={
   "offset":"2130404783", # Replace with a relevant offset or remove if just testing connectivity
    "limit":"1"
}
resp = requests.get(base_url,data=parameters)
print(resp.text)

## READ DATA FROM GROUP

This code cell demonstrates how to send a simple message to a specified Telegram chat ID. It initializes a `base_url` for the Telegram API `sendMessage` endpoint and constructs a `parameters` dictionary with the chat ID and message text.

In [ ]:
import requests
import time
from google.colab import userdata

# Retrieve Telegram Bot Token from Colab secrets
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
    if not TELEGRAM_BOT_TOKEN:
        raise ValueError("TELEGRAM_BOT_TOKEN found but is empty. Please ensure your bot token is correctly set.")
    print("Successfully loaded TELEGRAM_BOT_TOKEN for sendMessage test.")
except userdata.SecretNotFoundError:
    raise ValueError("TELEGRAM_BOT_TOKEN not found. Please add your Telegram bot token to Colab secrets.")

base_url=f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

bahasa = "Hello from Sahabat UMS chatbot!" # Define the 'bahasa' variable with a string value

parameters = {
   "chat_id":"YOUR_TELEGRAMBOT_ID", # IMPORTANT: Replace with your actual chat_id or fetch dynamically. DO NOT hardcode for public sharing.
    "text":bahasa
}
resp = requests.get(base_url,data=parameters)
print(resp.text)

This cell installs and imports necessary libraries for the Telegram bot and the Google Gemini API. It sets up `google.generativeai` for interacting with Gemini, `google.colab.userdata` for securely managing API keys, `requests` for HTTP communication with Telegram, and `json` for handling JSON data.

In [ ]:
# Install necessary libraries for Telegram bot and Google Gemini API
!pip install google-generativeai

# Import necessary libraries
import google.generativeai as genai
from google.colab import userdata # For securely storing API key
import requests
import time
import json

### Gemini API Configuration

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio:
1.  Go to [Google AI Studio](https://aistudio.google.com/app/apikey).
2.  Click on 'Create API key in new project'.

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Replace YOUR_API_KEY_HERE with yours. Then, the code below will securely retrieve and configure the key.

This code cell configures the Google Gemini API. It securely loads the primary API key from Colab secrets, ensuring only one key is used for all API calls. It then initializes the `genai` client with this key.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Load only the primary API key from Colab secrets
API_KEYS = []
try:
    key = userdata.get('GOOGLE_API_KEY')
    if key:
        API_KEYS.append(key)
        print("Successfully loaded GOOGLE_API_KEY")
    else:
        raise ValueError("GOOGLE_API_KEY found but is empty. Please ensure your API key is correctly set.")
except userdata.SecretNotFoundError:
    raise ValueError("GOOGLE_API_KEY not found. Please add your primary API key to Colab secrets.")

# Initialize the current API key and index (always 0 since there's only one key)
current_api_key_index = 0
GOOGLE_API_KEY = API_KEYS[current_api_key_index]
genai.configure(api_key=GOOGLE_API_KEY)
print("Gemini API key configured: Using the primary key.")

This cell simply prints the number of API keys that were successfully loaded. It serves as a quick check to confirm that the API key setup in the previous cell worked as expected.

In [ ]:
print(f"Number of API keys loaded: {len(API_KEYS)}")
# print(API_KEYS) # Uncomment with caution: this would display your actual API keys.

This cell defines a detailed `ums_glossary` as a system instruction for the Gemini model. This glossary guides the model to perform specialized English-Indonesian translations, focusing on academic and campus terminology specific to Universitas Muhammadiyah Surakarta (UMS). It also sets a low `temperature` for deterministic translation.

In [29]:
import google.generativeai as genai

ums_glossary = """
You are an expert English-Indonesian translator specializing in academic and campus contexts.
When translating, you must map specific Universitas Muhammadiyah Surakarta (UMS) terminology accurately.
Use the following domain dataset for contextual mapping and explicit translations:

--- CORE UMS DIGITAL PLATFORMS & PORTALS ---
- UMS STAR / STAR UMS: Sistem Terpadu Akademik Reguler. Translate as "STAR UMS (Integrated Academic Portal)". It handles course scheduling, grades (KHS), and SKS billing.
- SPADA UMS: Sistem Pembelajaran Daring UMS. Translate as "SPADA UMS (E-Learning Platform)". Used for online assignments, exams, and lecture materials.
- MyUMS: The unified single sign-on app or student hub portal.
- CAS UMS: Central Authentication Service. The secure single sign-on system used to log into SPADA and STAR.

--- ACADEMIC ADMIN & COURSEWORK REGISTRATION ---
- SKS: Satuan Kredit Semester. Translate as "SKS (Semester Credit Units)". Represents course weight.
- KRS: Kartu Rencana Studi. Translate as "KRS (Study Plan Card)". The selection of courses chosen by a student at the start of a semester.
- KHS: Kartu Hasil Studi. Translate as "KHS (Academic Progress Report / Semester Grades)".
- IPK: Indeks Prestasi Kumulatif. Translate as "GPA (Cumulative Grade Point Average)".
- Matakuliah / Makul: Course or Academic Subject.
- Cuti Akademik: Academic Leave / Deferment.

--- POSTGRADUATE & ADVANCED RESEARCH TERMS ---
- Tesis: Master's Thesis. (Note: Do not translate as general paper or bachelor thesis; it specifically refers to Master's level research).
- Skripsi: Bachelor's Thesis.
- Dosen Pembimbing: Academic Supervisor / Thesis Advisor.
- Dosen Pembimbing Utama: Primary Thesis Supervisor.
- Co-author: Penulis pendamping / Anggota penulis (for joint research journal publications).
- Publikasi Jurnal: Journal Publication (often tied to SINTA accreditation guidelines for graduation).

--- GENERAL CAMPUS LINGO & LOCATIONS ---
- Kampus 1, Kampus 2, Kampus 3: Campus 1, Campus 2, Campus 3 (The main geographic zones of UMS in Pabelan/Kartasura).
- Masa Registrasi: Registration / Enrollment Window.
- Pembayaran UKT / SKS: Tuition Fee / Credit Unit Payment.
---
If an English message mentions a portal like "STAR" or "SPADA", do not translate the name literally; keep it as "UMS STAR" or "SPADA UMS". If an Indonesian slang or abbreviation is used (e.g., "makul", "krsan"), resolve the typo or abbreviation using the context above before translating.
Also, attempt to correct any misspelled words or infer meaning for words that have no obvious meaning based on the context.
"""

config = genai.GenerationConfig(
    temperature=0.1  # Set very low for deterministic, exact translations
)

### Initialize Gemini Model

Before making any API calls, you need to initialize the Generative Model. I will use `gemini-2.5-flash` as it's suitable for text generation tasks like translation.

This code cell initializes two Gemini models: `gemini_model` for translation, configured with the `ums_glossary` as a system instruction, and `gemini_quiz_model` for quiz generation, without a specific system instruction.

In [30]:
import google.generativeai as genai

# Initialize the Gemini model for translation with UMS glossary
gemini_model = genai.GenerativeModel('gemini-2.5-flash', system_instruction=ums_glossary)

# Initialize a separate Gemini model for quiz generation
gemini_quiz_model = genai.GenerativeModel('gemini-2.5-flash') # No system_instruction for quiz

### Quiz Data and Functions

This code cell defines functions for managing a vocabulary quiz. `start_quiz` initiates a quiz by selecting a random Indonesian-English word pair and asking the user for a translation. `process_quiz_answer` evaluates the user's response, provides feedback, and `send_instant_quiz` triggers a quiz after a certain number of translations.

In [31]:
import random

# Dictionary to store ongoing quiz states for each chat_id
user_quiz_state = {}

def start_quiz(chat_id, send_message_callback):
    """Starts a direct-input vocabulary quiz based on the basic Indonesian dataset."""
    print(f"[DEBUG] Entering start_quiz for chat_id: {chat_id}")
    global user_quiz_state

    if not BASIC_INDONESIAN_WORDS:
        send_message_callback(chat_id, "Sorry, no quiz questions are available right now.")
        return

    # 1. Pick a random category and word
    random_category_key = random.choice(list(BASIC_INDONESIAN_WORDS.keys()))
    random_item = random.choice(BASIC_INDONESIAN_WORDS[random_category_key])

    # 2. Extract words (cleaning up slashes if there are multiple options)
    indonesian_word_base = random_item['indonesian'].split('/')[0].strip()
    english_word_base = random_item['english'].split('/')[0].strip()

    # 3. Setup the quiz direction (Ask in English, expect Indonesian)
    word_to_ask = english_word_base
    correct_translation = indonesian_word_base

    # 4. Format the direct question
    full_question_message = (
        f"** Quiz!**\n\n"
        f"What is the Indonesian translation for: **'{word_to_ask}'**?\n\n"
        f"*(Type your answer below)*"
    )

    # 5. Save the state so the bot knows what answer to look for
    user_quiz_state[chat_id] = {
        'word_to_ask': word_to_ask,
        'correct_translation': correct_translation.lower()
    }

    send_message_callback(chat_id, full_question_message)
    print(f"[DEBUG] Quiz question sent for chat_id: {chat_id}")


def process_quiz_answer(chat_id, user_answer_text, send_message_callback):
    """Evaluates the user's typed text against the correct translation."""
    global user_quiz_state
    current_quiz = user_quiz_state.get(chat_id)

    if not current_quiz:
        return False # Not in a quiz state

    correct_translation = current_quiz['correct_translation']

    # Normalize the user's answer (remove extra spaces and make lowercase)
    user_answer_norm = user_answer_text.strip().lower()

    # Check if the typed answer matches the correct translation
    if user_answer_norm == correct_translation:
        send_message_callback(chat_id, "✅ **Correct!** Great job!")
    else:
        send_message_callback(chat_id, f"❌ **Incorrect.**\nThe correct answer was: **{correct_translation.title()}**")

    # End the quiz state for this user
    del user_quiz_state[chat_id]

    # Send a follow-up prompt
    send_message_callback(chat_id, "Type `/quiz` for another question, or just send a message to translate it!")
    return True


def send_instant_quiz(chat_id, send_message_callback):
    """Triggers the basic vocabulary quiz immediately after a translation."""
    # Since we abandoned the UMS Gemini quiz, we just call start_quiz directly!
    start_quiz(chat_id, send_message_callback)

This cell initializes `BASIC_INDONESIAN_WORDS`, a dictionary containing simple Indonesian-English word pairs categorized for use in the quiz functionality. It serves as the dataset for generating quiz questions.

In [32]:
BASIC_INDONESIAN_WORDS = {
    "Basic Greetings": [
        {"indonesian": "Halo", "english": "Hello"},
        {"indonesian": "Terima kasih", "english": "Thank you"}
    ],
    "Common Phrases": [
        {"indonesian": "Apa kabar?", "english": "How are you?"},
        {"indonesian": "Selamat pagi", "english": "Good morning"}
    ]
}
print(BASIC_INDONESIAN_WORDS)

{'Basic Greetings': [{'indonesian': 'Halo', 'english': 'Hello'}, {'indonesian': 'Terima kasih', 'english': 'Thank you'}], 'Common Phrases': [{'indonesian': 'Apa kabar?', 'english': 'How are you?'}, {'indonesian': 'Selamat pagi', 'english': 'Good morning'}]}


### Telegram Chatbot Logic

Now, let's set up the main logic for the Telegram chatbot. This code will continuously poll for new messages sent to the bot, translate any Indonesian text it receives to English using the Google Cloud Translation API, and then send the translation back to the user.

**Note**: You can find the Telegram bot token from the `base_url` variable in the existing cells.

To stop the bot, you will need to interrupt the execution of the cell (e.g., by clicking the stop button in Colab).

This is the main loop for the Telegram bot. It continuously polls for new messages using `getUpdates`, processes commands like `/start` and `/quiz`, and handles regular translation requests. It detects the source language, constructs a prompt for the Gemini API (including error correction and example sentences), and sends the translated response back to the user. It also manages translation counters and triggers instant quizzes.

In [ ]:
import requests
import time
from google.colab import userdata

# Helper function to send messages (used by quiz functions)
def send_telegram_message(chat_id, text):
    send_message_url = f"{TELEGRAM_API_BASE_URL}/sendMessage"
    send_params = {
        "chat_id": chat_id,
        "text": text
        # Removed "parse_mode": "Markdown" to rule out formatting issues
    }
    try:
        requests.post(send_message_url, json=send_params).raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Failed to send message to {chat_id}: {e}")

# --- New function to determine sentence complexity ---
def is_simple_sentence(text):
    """
    A simple heuristic to determine if a sentence is 'simple'.
    Criteria can be adjusted (e.g., word count, presence of complex keywords).
    For this example, we'll consider sentences with <= 5 words as simple.
    """
    words = text.split()
    return len(words) <= 5

# --- BEGIN MODIFIED BOT LOOP ---
# Retrieve Telegram Bot Token from Colab secrets
try:
    TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
    if not TELEGRAM_BOT_TOKEN:
        raise ValueError("TELEGRAM_BOT_TOKEN found but is empty. Please ensure your bot token is correctly set.")
    print("Successfully loaded TELEGRAM_BOT_TOKEN for bot loop.")
except userdata.SecretNotFoundError:
    raise ValueError("TELEGRAM_BOT_TOKEN not found. Please add your Telegram bot token to Colab secrets.")

TELEGRAM_API_BASE_URL = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}"

last_update_id = 0
print("Starting Telegram translator bot... (Press stop to halt)")

# Define the welcome message
WELCOME_MESSAGE = "Hello! I'm your UMS academic translator bot. I can translate between English and Indonesian, especially for UMS-related academic terms. Try sending me a message or type /quiz to test your memory!"

# Dictionary to store translation counts per chat_id
translation_counters = {}

# --- Removed API Key Rotation Logic ---
# The primary API key is initialized in e9dd160b and will be used directly.

while True:
    try:
        get_updates_url = f"{TELEGRAM_API_BASE_URL}/getUpdates"
        params = {
            "offset": last_update_id + 1,
            "timeout": 30
        }

        resp = requests.get(get_updates_url, params=params)
        resp.raise_for_status()
        response_data = resp.json()

        if not response_data['ok']:
            print(f"Telegram API error: {response_data.get('description', 'Unknown error')}")
            time.sleep(5)
            continue

        updates = response_data['result']

        for update in updates:
            last_update_id = update['update_id']
            if 'message' in update:
                message = update['message']
                chat_id = message['chat']['id']
                text = message.get('text')

                # Initialize these for each message to avoid NameError
                is_unrecognizable = False
                corrected_output = ""
                translated_output = ""
                example_output = ""

                # Handle /start or new chat members
                if 'new_chat_members' in message or (text and text.lower() == '/start'):
                    send_telegram_message(chat_id, WELCOME_MESSAGE)
                    print(f"Sent welcome message to chat {chat_id}")
                    continue

                # Handle /quiz command
                if text and text.lower() == '/quiz':
                    start_quiz(chat_id, send_telegram_message)
                    print(f"Started quiz for chat {chat_id}")
                    continue

                # Handle /perbaiki or /feedback command for linguistic evaluation
                if text and (text.lower().startswith('/perbaiki') or text.lower().startswith('/feedback')):
                    text_to_evaluate = text[text.find(' ')+1:].strip()
                    # Assuming handle_feedback exists and uses gemini_feedback_model
                    # You might need to ensure gemini_feedback_model and feedback_config are initialized if this feature is desired.
                    # For this context, it's commented out as its setup was previously suggested for deletion.
                    # handle_feedback(chat_id, text_to_evaluate, send_telegram_message)
                    print(f"Processed feedback request for chat {chat_id}: '{text_to_evaluate}' (Feedback functionality requires gemini_feedback_model setup)")
                    send_telegram_message(chat_id, "Feedback functionality is not active. Please set up `gemini_feedback_model` if you wish to use it.")
                    continue

                # Process quiz answers if user is in a quiz state
                if chat_id in user_quiz_state:
                    if process_quiz_answer(chat_id, text, send_telegram_message):
                        print(f"Processed quiz answer for chat {chat_id}: {text}")
                        # Reset translation counter after a quiz if user was actively translating
                        if chat_id in translation_counters:
                            translation_counters[chat_id] = 0
                        continue

                # Regular translation logic
                if text:
                    print(f"Received message from chat {chat_id}: {text}")

                    source_lang_detected = 'id'
                    target_lang_detected = 'en'

                    english_keywords = ['hello', 'thank you', 'good morning', 'welcome', 'how are you', 'what is']
                    indonesian_keywords = ['halo', 'terima kasih', 'selamat pagi', 'selamat datang', 'apa kabar', 'apa itu']

                    text_lower = text.lower()

                    is_english = any(kw in text_lower for kw in english_keywords)
                    is_indonesian = any(kw in text_lower for kw in indonesian_keywords)

                    if is_english and not is_indonesian:
                        source_lang_detected = 'en'
                        target_lang_detected = 'id'
                        print(f"Detected English for '{text}' (translating to Indonesian)")
                    elif is_indonesian and not is_english:
                        source_lang_detected = 'id'
                        target_lang_detected = 'en'
                        print(f"Detected Indonesian for '{text}' (translating to English)")
                    else:
                        source_lang_detected = 'id'
                        target_lang_detected = 'en'
                        print(f"Could not clearly detect language for '{text}', defaulting to Indonesian -> English.")

                    final_response_message = ""
                    translation_successful = False

                    # Attempt translation with the single API key
                    print(f"Attempting translation with the primary API key.")

                    try:
                        if 'gemini_model' not in globals():
                            raise ValueError("Gemini model not initialized. Please run the Gemini configuration cell.")

                        prompt = (
                            f"First, correct any typos or grammatical errors in the following {source_lang_detected} text. "
                            f"If a correction is made, include the corrected text in the response. "
                            f"Then, translate the corrected text to {target_lang_detected}. "
                            f"Also, provide an example sentence in {target_lang_detected} that relates to the meaning of the original {source_lang_detected} word/phrase '{text}'. "
                            f"Format your response as follows:\n\n"
                            f"Correction: [corrected text, if any]\n"
                            f"Translation: [translated text]\n"
                            f"Example: [example sentence]\n\n"
                            f"Text to translate: \"{text}\"")

                        gemini_response = gemini_model.generate_content(prompt, generation_config=config)
                        full_response_text = gemini_response.text

                        # Parse the response for correction, translation, and example
                        correction_prefix = "Correction: "
                        translation_prefix = "Translation: "
                        example_prefix = "Example: "
                        corrected_output = ""
                        translated_output = ""
                        example_output = ""

                        lines = full_response_text.split('\n')
                        for line in lines:
                            if line.startswith(correction_prefix):
                                corrected_output = line[len(correction_prefix):].strip()
                            elif line.startswith(translation_prefix):
                                translated_output = line[len(translation_prefix):].strip()
                            elif line.startswith(example_prefix):
                                example_output = line[len(example_prefix):].strip()

                        response_parts = []
                        is_unrecognizable = False # Reset for Gemini path

                        # Check if Gemini identified the word as unrecognizable
                        if "not a recognizable word or phrase" in translated_output.lower() or \
                            "no meaningful correction can be made" in corrected_output.lower():
                            is_unrecognizable = True
                            final_response_message = "unrecognizable phrase, please try again."
                        else:
                            if corrected_output and corrected_output.lower() != text.lower():
                                response_parts.append(f"_Typo detected! Corrected from: '{text}' to '{corrected_output}'_\n")

                            if translated_output:
                                response_parts.append(f"*Translation:* {translated_output}")
                                if example_output:
                                    response_parts.append(f"*Example:* {example_output}")
                            else:
                                response_parts.append("Sorry, I couldn't get a clear translation from Gemini.")

                            final_response_message = '\n'.join(response_parts)

                        print(f"Translated '{text}' (from {source_lang_detected} to {target_lang_detected}) to '{translated_output}' with example '{example_output}' using Gemini")
                        translation_successful = True
                    except Exception as e:
                        print(f"Translation error with the primary key: {e}")
                        if "You exceeded your current quota" in str(e): # Updated condition for quota error
                            # Send immediate feedback to the user about the quota
                            send_telegram_message(chat_id, "You have reached limits for the free tier for this API key. Please wait or upgrade your plan.")
                            print(f"Sent quota message to chat {chat_id}.")

                            # Sleep for 60 seconds as requested by the user after hitting quota (optional, but good for user experience if they can just wait for one key)
                            print("Pausing for 60 seconds due to quota exceedance for the single API key...")
                            time.sleep(60)
                            print("Resuming after 60 seconds.")
                            final_response_message = "This API key has hit its quota limit. Please wait or upgrade your plan."
                            translation_successful = False
                        else:
                            # Non-quota error, set a generic message
                            final_response_message = "An unexpected error occurred during translation. Please try again later."
                            translation_successful = False

                    # Send the final response or error message if one has been set.
                    if final_response_message:
                        send_telegram_message(chat_id, final_response_message)
                        print(f"Sent message to chat {chat_id}: {final_response_message}")

                    # Only increment counter and trigger quiz if translation was actually successful AND not unrecognizable
                    if translation_successful and not is_unrecognizable:
                            translation_counters[chat_id] = translation_counters.get(chat_id, 0) + 1

                            # Trigger instant quiz after every 3 translations
                            if translation_counters[chat_id] % 3 == 0:
                                send_instant_quiz(chat_id, send_telegram_message)
                                print(f"Triggered quiz for chat {chat_id} after {translation_counters[chat_id]} translations.")

            elif 'new_chat_members' in update['message']:
                chat_id = update['message']['chat']['id']
                send_telegram_message(chat_id, WELCOME_MESSAGE)
                print(f"Sent welcome message to new chat members in chat {chat_id}")

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    time.sleep(1) # Poll every 1 second to prevent excessive API calls